# 📈 Linear Regression — A Step-by-Step Guide

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/linear_regression.ipynb)

## What is Linear Regression?

Linear Regression is one of the oldest and most widely used algorithms in machine learning and statistics. At its heart, it answers a simple question:

> **"Given some input values, can I predict a continuous output value?"**

For example:
- Given the **size of a house**, can I predict its **price**?
- Given a **patient's age and BMI**, can I predict their **medical expenses**?

---

## 🔢 The Math Behind It

The simplest form — **Simple Linear Regression** — models the relationship between one input feature $x$ and one output $y$ using a straight line:

$$\hat{y} = \beta_0 + \beta_1 x$$

Where:
- $\hat{y}$ → predicted output
- $\beta_0$ → **intercept** (where the line crosses the y-axis)
- $\beta_1$ → **slope** (how much $y$ changes per unit increase in $x$)

When we have **multiple features** ($x_1, x_2, ..., x_n$), this extends to:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

---

## 🎯 What Does "Fitting" Mean?

The model **learns** by finding the values of $\beta$ (coefficients) that minimize the **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

This is called the **Ordinary Least Squares (OLS)** method — we're literally minimizing the squared distances between actual and predicted values.

---

## ⚠️ A Critical Assumption: Correlation

Linear Regression **assumes** a linear relationship between features and the target. If your features have **no correlation** with the target, the model will perform poorly — garbage in, garbage out.

This is why **Exploratory Data Analysis (EDA)** always comes before modelling. We need to *see* whether fitting a line even makes sense.

Let's now build this up step by step. 🚀

---
## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# Keep plots neat
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print("✅ All libraries imported successfully!")

---
## Step 2 — Load the Dataset

We'll use scikit-learn's built-in **Diabetes dataset**.

### About the Dataset

The dataset contains **442 patient records** collected from diabetes patients. Each record has:
- **10 features**: age, sex, BMI, average blood pressure, and 6 blood serum measurements (all normalized)
- **Target**: a quantitative measure of **disease progression** one year after baseline (a continuous value)

This makes it a perfect fit for regression. Our goal: predict disease progression from clinical measurements.

In [ ]:
# Load dataset
diabetes = load_diabetes()

# Create a DataFrame for easier handling
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {diabetes.feature_names}")
print(f"\nTarget: disease progression score")
df.head()

---
## Step 3 — Exploratory Data Analysis (EDA)

Before we model anything, we explore the data. This step is **not optional** — it tells us:
1. What the data looks like (distribution of features and target)
2. Whether features are **correlated** with the target (essential for Linear Regression)
3. Whether there are obvious outliers or data quality issues

### 3.1 Basic Statistics

In [ ]:
df.describe().round(3)

### 3.2 Target Distribution

Let's see what we're trying to predict — the distribution of disease progression scores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['target'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Target (Disease Progression)', fontweight='bold')
axes[0].set_xlabel('Disease Progression Score')
axes[0].set_ylabel('Count')
axes[0].axvline(df['target'].mean(), color='red', linestyle='--', label=f"Mean = {df['target'].mean():.1f}")
axes[0].legend()

# Boxplot
axes[1].boxplot(df['target'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot of Target', fontweight='bold')
axes[1].set_ylabel('Disease Progression Score')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['target'])

plt.suptitle('Target Variable Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"Mean: {df['target'].mean():.2f}  |  Std: {df['target'].std():.2f}  |  Min: {df['target'].min()}  |  Max: {df['target'].max()}")

### 3.3 Correlation — The Heart of Linear Regression

> **Why correlation matters so much?**
>
> Linear Regression works by drawing the *best fitting line* between features and the target. For this line to be meaningful, the features must **move together** with the target in a consistent direction — that's correlation.
>
> - A correlation close to **+1** → strong positive linear relationship
> - A correlation close to **-1** → strong negative linear relationship
> - A correlation close to **0** → no linear relationship — the feature is not useful for Linear Regression

Let's visualize the **correlation matrix** first.

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle (redundant)
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    mask=mask,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Matrix — All Features + Target', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.4 Feature vs Target Correlation (Sorted)

Let's zoom in on what matters most — **how each feature correlates with the target**.

In [ ]:
target_corr = df.corr()['target'].drop('target').sort_values()

colors = ['tomato' if v < 0 else 'steelblue' for v in target_corr.values]

plt.figure(figsize=(10, 5))
bars = plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white', height=0.6)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.axvline(0.3, color='green', linewidth=1, linestyle=':', alpha=0.7, label='|r| = 0.3 threshold')
plt.axvline(-0.3, color='green', linewidth=1, linestyle=':', alpha=0.7)
plt.xlabel('Pearson Correlation with Target')
plt.title('Feature Correlation with Disease Progression (Target)', fontweight='bold')
plt.legend()

# Annotate values
for bar, val in zip(bars, target_corr.values):
    plt.text(val + 0.01 * np.sign(val), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()

print("\nCorrelation values with target:")
print(target_corr.round(3).to_string())

### 3.5 Scatter Plots — Visualizing the Linear Relationship

Numbers are good, but **scatter plots** show us the *shape* of the relationship. If the cloud of points roughly follows a line — we're good to go!

In [ ]:
features = diabetes.feature_names
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feature in enumerate(features):
    corr_val = df[feature].corr(df['target'])
    color = 'steelblue' if abs(corr_val) >= 0.3 else 'lightgray'
    
    axes[i].scatter(df[feature], df['target'], alpha=0.4, s=20, color=color)
    
    # Regression line
    m, b = np.polyfit(df[feature], df['target'], 1)
    x_line = np.linspace(df[feature].min(), df[feature].max(), 100)
    axes[i].plot(x_line, m * x_line + b, color='red', linewidth=1.5)
    
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Target' if i % 5 == 0 else '')
    axes[i].set_title(f'{feature}\nr = {corr_val:.3f}', fontweight='bold',
                      color='green' if abs(corr_val) >= 0.3 else 'gray')

plt.suptitle('Scatter Plots: Each Feature vs Target  (green title = strong correlation)', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 🔍 EDA Summary

From our exploration:
- **`bmi`** (r = 0.586) and **`s5`** (r = 0.565) have the **strongest positive correlations** with the target
- **`s1`** and **`s2`** show **negative correlations** — as these features increase, the disease progression tends to decrease
- **`sex`** has the weakest correlation — not very informative on its own

The scatter plots confirm that most features show a **rough linear trend** with the target — so Linear Regression is a reasonable choice here!

---

## Step 4 — Prepare Features and Target

In [ ]:
X = df.drop(columns='target')   # Feature matrix
y = df['target']                 # Target vector

print(f"Feature matrix X: {X.shape}  → {X.shape[0]} samples × {X.shape[1]} features")
print(f"Target vector y:  {y.shape}  → {y.shape[0]} samples")

---
## Step 5 — Fit on the Entire Dataset (No Split)

> **Wait — why are we fitting on ALL the data first?**
>
> We're doing this **intentionally** as a teaching step.
>
> When you train on 100% of the data, the model can squeeze every bit of pattern out of it — including patterns that are just noise or quirks of *this particular dataset*. The metrics will look great. But the model hasn't been **tested** on data it has never seen. So we don't really know if it *generalizes*.
>
> We'll come back to this important point later and show why splitting is necessary.

For now, let's fit and see the results.

In [ ]:
# Initialize and train the model
model_full = LinearRegression()
model_full.fit(X, y)

# Predict on the same data used for training
y_pred_full = model_full.predict(X)

print("Model trained on full dataset.")
print(f"\nIntercept (β₀): {model_full.intercept_:.4f}")
print(f"\nCoefficients (β₁ ... βₙ):")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_full.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

### 5.1 Evaluate — Full Dataset Metrics

#### Metrics Glossary

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | $\frac{1}{n}\sum|y - \hat{y}|$ | Average absolute error. In same units as target. |
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | Penalizes large errors more heavily. |
| **RMSE** | $\sqrt{\text{MSE}}$ | Square root of MSE. Comparable to target scale. |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained. 1.0 = perfect. |

In [ ]:
mae_full  = mean_absolute_error(y, y_pred_full)
mse_full  = mean_squared_error(y, y_pred_full)
rmse_full = np.sqrt(mse_full)
r2_full   = r2_score(y, y_pred_full)

print("=" * 45)
print("  Metrics — Trained & Evaluated on FULL Data")
print("=" * 45)
print(f"  MAE  : {mae_full:.4f}")
print(f"  MSE  : {mse_full:.4f}")
print(f"  RMSE : {rmse_full:.4f}")
print(f"  R²   : {r2_full:.4f}")
print("=" * 45)

### 5.2 Visualize — Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y, y_pred_full, alpha=0.5, color='steelblue', s=25, label='Predictions')
min_val, max_val = y.min(), y.max()
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Fit (y = x)')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Actual vs Predicted\n(Full Data | R² = {r2_full:.3f})', fontweight='bold')
axes[0].legend()

# Residuals
residuals_full = y - y_pred_full
axes[1].scatter(y_pred_full, residuals_full, alpha=0.5, color='darkorange', s=25)
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals (Actual − Predicted)')
axes[1].set_title('Residual Plot\n(Full Data)', fontweight='bold')

plt.suptitle('Model Performance — Full Dataset Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6 — Why the Full-Data Approach is Problematic 🚨

The metrics above look decent. But here's the catch:

> **We trained and tested on the same data.**
>
> This is like studying for an exam using *the actual exam paper*, then claiming you've mastered the subject. Of course you'll score well — but you haven't actually learned to generalise!

### What Could Go Wrong?

- The model might have **memorised** patterns specific to this dataset — called **overfitting**
- When given **new, unseen data**, it may perform significantly worse
- The R² score we computed is **optimistically biased** — it flatters the model

### The Solution — Train/Test Split

We split the data into two non-overlapping sets:

| Set | Purpose | Typical Size |
|-----|---------|-------------|
| **Training set** | Model learns from this | 80% |
| **Test set** | Evaluate generalization — model never sees this during training | 20% |

The test set **simulates real-world, unseen data**. If the model performs well on it, we can trust it.

---
## Step 7 — Proper 80/20 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% goes to test set
    random_state=42     # Fix seed for reproducibility
)

print(f"Training set   : {X_train.shape[0]} samples  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set       : {X_test.shape[0]}  samples  ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nTotal samples  : {len(X)}")

---
## Step 8 — Fit on Training Data Only

In [ ]:
model_split = LinearRegression()
model_split.fit(X_train, y_train)   # ← Only train data goes here

# Predict on both train and test sets
y_pred_train = model_split.predict(X_train)
y_pred_test  = model_split.predict(X_test)   # ← Unseen data

print("Model trained on training set only.")
print(f"Intercept: {model_split.intercept_:.4f}")

---
## Step 9 — Compare Train vs Test Performance

The key insight: if **training metrics >> test metrics**, the model has overfit — it memorised instead of learning.

In [ ]:
def compute_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    return {'Set': label, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}

results = pd.DataFrame([
    compute_metrics(y,       y_pred_full,  'Full Data (No Split)'),
    compute_metrics(y_train, y_pred_train, 'Train Set (80%)'),
    compute_metrics(y_test,  y_pred_test,  'Test Set  (20%)'),
]).set_index('Set').round(4)

print(results.to_string())

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [
    (y,       y_pred_full,  'Full Data (No Split)',  f'R² = {r2_score(y, y_pred_full):.3f}',   'steelblue'),
    (y_train, y_pred_train, 'Train Set (80%)',        f'R² = {r2_score(y_train, y_pred_train):.3f}', 'mediumseagreen'),
    (y_test,  y_pred_test,  'Test Set (20%) — Unseen', f'R² = {r2_score(y_test, y_pred_test):.3f}',  'tomato'),
]

for ax, (y_true, y_pred, title, r2_label, color) in zip(axes, datasets):
    ax.scatter(y_true, y_pred, alpha=0.5, color=color, s=25)
    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1.5, label='Perfect Fit')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{title}\n{r2_label}', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Actual vs Predicted — Comparing Evaluation Strategies', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 10 — Coefficient Interpretation

The coefficients tell us how much the prediction changes for a **one-unit increase** in each feature (holding others constant).

Since features in the Diabetes dataset are already **normalized** (zero mean, unit variance), coefficients are directly comparable.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_split.coef_
}).sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['Coefficient']]

plt.figure(figsize=(10, 5))
bars = plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white', height=0.6)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title('Linear Regression Coefficients\n(Model trained on 80% split)', fontweight='bold')

for bar, val in zip(bars, coef_df['Coefficient']):
    plt.text(val + 2 * np.sign(val), bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()

print("\nTop positive drivers (increase disease progression):")
print(coef_df[coef_df['Coefficient'] > 0].sort_values('Coefficient', ascending=False).to_string(index=False))
print("\nTop negative drivers (decrease disease progression):")
print(coef_df[coef_df['Coefficient'] < 0].sort_values('Coefficient').to_string(index=False))

---
## Step 11 — Residual Analysis

A well-behaved linear regression should have residuals that are:
- **Randomly scattered** around zero (no pattern)
- **Approximately normally distributed**
- **Constant variance** (homoscedastic)

If residuals show a pattern, it means the model is missing some structure in the data.

In [ ]:
residuals_test = y_test - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Residuals vs Predicted
axes[0].scatter(y_pred_test, residuals_test, alpha=0.5, color='darkorange', s=25)
axes[0].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted\n(Test Set)', fontweight='bold')

# 2. Histogram of residuals
axes[1].hist(residuals_test, bins=25, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution\n(Test Set)', fontweight='bold')

# 3. Q-Q Plot
from scipy import stats
stats.probplot(residuals_test, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot of Residuals\n(Should Follow the Line)', fontweight='bold')
axes[2].get_lines()[1].set_color('red')

plt.suptitle('Residual Analysis — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📋 Final Summary

Here's what we built and learned in this notebook:

### 🧱 The Building Blocks

| Step | What We Did | Why It Matters |
|------|-------------|----------------|
| 1 | Imported libraries | Numpy, Pandas, Sklearn, Seaborn are our toolkit |
| 2 | Loaded the Diabetes dataset | A clean, real-world regression dataset |
| 3 | EDA — distributions, correlations, scatter plots | **Correlation must exist** for Linear Regression to make sense |
| 4 | Prepared X and y | Separated features from the target |
| 5 | Fit on full dataset | Demonstrates the "training bias" trap |
| 6 | Understood the problem | Training = testing → overly optimistic metrics |
| 7 | 80/20 Train-Test Split | Proper way to evaluate generalization |
| 8 | Fit on training set | Model only learns from 80% |
| 9 | Compared metrics | Full data metrics vs honest test set metrics |
| 10 | Coefficient interpretation | Which features matter most |
| 11 | Residual analysis | Checks model assumptions |

### 🔑 Key Takeaways

1. **Correlation is the foundation** — always check it before fitting a linear model
2. **Never evaluate on training data alone** — it gives a falsely optimistic picture
3. **The test set is sacred** — it should never influence model training or selection
4. **Residual plots reveal hidden problems** — always check them after fitting
5. **R² on test data is what matters** — that's your real-world performance estimate